# U-Net34 — Evaluate Kaggle Input Validation + Top/Bottom 3 by Dice

Notebook này **không train lại model**.

Nó thực hiện:

1. Tự tìm official ISIC 2018 Validation trong `/kaggle/input`.
2. Tự tìm checkpoint model trong `/kaggle/input`.
   - Ưu tiên `unet34_selected_single_best.pt`.
   - Nếu không có, dùng các `unet34_best.pt` tìm được làm equal-weight ensemble.
3. Áp dụng đúng **Simplest Color Balance 1%** trước resize.
4. Inference ở **256×256 trên GPU T4**, AMP FP16.
5. Tính Dice / IoU (Jaccard) / Sensitivity theo từng ảnh.
6. Xuất CSV và visual:
   - Top 3 Dice tốt nhất.
   - Top 3 Dice tệ nhất.

In [ ]:
import math
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm

# -------------------- Config --------------------
KAGGLE_INPUT = Path("/kaggle/input")

# Có thể đặt đường dẫn cụ thể nếu /kaggle/input có nhiều bộ model.
# Ví dụ:
# MODEL_SEARCH_ROOT = Path("/kaggle/input/unet34-models")
MODEL_SEARCH_ROOT = KAGGLE_INPUT

VALIDATION_IMAGE_DIRNAME = "ISIC2018_Task1-2_Validation_Input"
VALIDATION_MASK_DIRNAME = "ISIC2018_Task1_Validation_GroundTruth"
EXPECTED_VALIDATION_COUNT = 100

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = min(2, os.cpu_count() or 1)
PRED_THRESHOLD = 0.5
COLOR_BALANCE_PERCENT = 1.0
USE_AMP = True
FP32_TRANSPOSED_CONV = True

OUTPUT_DIR = Path("/kaggle/working/validation_eval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU Accelerator trong Kaggle.")

DEVICE = torch.device("cuda:0")
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT_DIR)


In [ ]:
# -------------------- Exact Simplest Color Balance 1% --------------------
def apply_mask(matrix, mask, fill_value):
    masked = np.ma.array(matrix, mask=mask, fill_value=fill_value)
    return masked.filled()


def apply_threshold(matrix, low_value, high_value):
    low_mask = matrix < low_value
    matrix = apply_mask(matrix, low_mask, low_value)

    high_mask = matrix > high_value
    matrix = apply_mask(matrix, high_mask, high_value)

    return matrix


def simplest_cb(img, percent):
    assert img.shape[2] == 3, "shape is wrong"
    assert percent > 0 and percent < 100

    half_percent = percent / 200.0
    channels = cv2.split(img)

    out_channels = []
    for channel in channels:
        assert len(channel.shape) == 2

        height, width = channel.shape
        vec_size = width * height
        flat = channel.reshape(vec_size)

        assert len(flat.shape) == 1

        flat = np.sort(flat)
        n_cols = flat.shape[0]

        low_val = flat[math.floor(n_cols * half_percent)]
        high_val = flat[math.ceil(n_cols * (1.0 - half_percent))]

        thresholded = apply_threshold(channel, low_val, high_val)
        normalized = cv2.normalize(
            thresholded,
            thresholded.copy(),
            0,
            255,
            cv2.NORM_MINMAX,
        )
        out_channels.append(normalized)

    return cv2.merge(out_channels)


def read_color_balanced_rgb(path):
    # Code gốc Color Balance hoạt động trên ảnh cv2.imread() dạng BGR.
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(path)

    image_bgr = simplest_cb(image_bgr, COLOR_BALANCE_PERCENT)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return image_rgb


In [ ]:
# -------------------- Discover official validation --------------------
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def populated_dirs_named(root, directory_name, suffixes):
    candidates = []
    for path in root.rglob(directory_name):
        if not path.is_dir():
            continue

        # Một số Kaggle datasets có cấu trúc dir/dir/files.
        for candidate in (path / directory_name, path):
            if (
                candidate.is_dir()
                and any(
                    p.is_file() and p.suffix.lower() in suffixes
                    for p in candidate.iterdir()
                )
            ):
                candidates.append(candidate.resolve())

    # unique, deterministic
    return sorted(set(candidates), key=lambda p: (len(str(p)), str(p)))


image_candidates = populated_dirs_named(
    KAGGLE_INPUT,
    VALIDATION_IMAGE_DIRNAME,
    IMAGE_SUFFIXES,
)
mask_candidates = populated_dirs_named(
    KAGGLE_INPUT,
    VALIDATION_MASK_DIRNAME,
    {".png"},
)

if not image_candidates:
    raise FileNotFoundError(
        f"Không tìm thấy {VALIDATION_IMAGE_DIRNAME} dưới {KAGGLE_INPUT}"
    )
if not mask_candidates:
    raise FileNotFoundError(
        f"Không tìm thấy {VALIDATION_MASK_DIRNAME} dưới {KAGGLE_INPUT}"
    )

# Ưu tiên candidate có đúng 100 ảnh/mask.
def image_count(path):
    return sum(
        p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
        for p in path.iterdir()
    )


def mask_count(path):
    return sum(
        p.is_file()
        and p.suffix.lower() == ".png"
        and p.stem.endswith("_segmentation")
        for p in path.iterdir()
    )


valid_image_candidates = [
    p for p in image_candidates if image_count(p) == EXPECTED_VALIDATION_COUNT
]
valid_mask_candidates = [
    p for p in mask_candidates if mask_count(p) == EXPECTED_VALIDATION_COUNT
]

VAL_IMAGE_DIR = (
    valid_image_candidates[0] if valid_image_candidates else image_candidates[0]
)
VAL_MASK_DIR = (
    valid_mask_candidates[0] if valid_mask_candidates else mask_candidates[0]
)

image_paths = [
    p
    for p in VAL_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
]
mask_paths = [
    p
    for p in VAL_MASK_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() == ".png"
    and p.stem.endswith("_segmentation")
]

image_map = {p.stem: p for p in image_paths}
mask_map = {
    p.stem.removesuffix("_segmentation"): p
    for p in mask_paths
}

missing_masks = sorted(set(image_map) - set(mask_map))
orphan_masks = sorted(set(mask_map) - set(image_map))
if missing_masks or orphan_masks:
    raise RuntimeError(
        f"Validation image/mask mismatch. "
        f"missing_masks={missing_masks[:5]}, orphan_masks={orphan_masks[:5]}"
    )

validation_records = [
    {
        "image_id": image_id,
        "image_path": image_map[image_id],
        "mask_path": mask_map[image_id],
    }
    for image_id in sorted(image_map)
]

if len(validation_records) != EXPECTED_VALIDATION_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_VALIDATION_COUNT} validation pairs, "
        f"found {len(validation_records)}"
    )

print("Validation images:", VAL_IMAGE_DIR)
print("Validation masks :", VAL_MASK_DIR)
print("Validation pairs :", len(validation_records))


In [ ]:
# -------------------- Validation Dataset --------------------
class ValidationDataset(Dataset):
    def __init__(self, records, image_size=256):
        self.records = list(records)
        self.image_size = int(image_size)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        image = read_color_balanced_rgb(record["image_path"])
        mask = cv2.imread(str(record["mask_path"]), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(record["mask_path"])

        if image.shape[:2] != mask.shape:
            raise ValueError(
                f"Image/mask shape mismatch: {record['image_id']}: "
                f"{image.shape[:2]} vs {mask.shape}"
            )

        mask = (mask > 127).astype(np.uint8)

        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR,
        )
        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST,
        )

        # HWC uint8 -> CHW uint8
        image_tensor = torch.from_numpy(
            np.ascontiguousarray(image.transpose(2, 0, 1))
        )
        mask_tensor = torch.from_numpy(mask[None].astype(np.float32))

        return image_tensor, mask_tensor, record["image_id"]


validation_loader = DataLoader(
    ValidationDataset(validation_records, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)

print("Validation batches:", len(validation_loader))


In [ ]:
# -------------------- Exact U-Net34 architecture --------------------
class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels // 2,
            2,
            stride=2,
        )
        self.skip_projection = nn.Conv2d(
            skip_channels,
            out_channels // 2,
            1,
        )
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x, skip):
        source_dtype = x.dtype

        if FP32_TRANSPOSED_CONV and x.is_cuda:
            with torch.autocast(device_type="cuda", enabled=False):
                x = self.up(x.float())
            x = x.to(dtype=source_dtype)
        else:
            x = self.up(x)

        skip = self.skip_projection(skip)

        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="nearest",
            )

        return self.bn(F.relu(torch.cat([x, skip], dim=1)))


class UNet34(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet34(weights=None)

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
        )
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 256)
        self.up2 = UpBlock(256, 64, 256)
        self.up1 = UpBlock(256, 64, 256)
        self.up0 = UpBlock(256, 3, 16)

        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)

        bottleneck = self.layer4(s3)

        decoded = self.up4(bottleneck, s3)
        decoded = self.up3(decoded, s2)
        decoded = self.up2(decoded, s1)
        decoded = self.up1(decoded, s0)
        decoded = self.up0(decoded, x)

        return self.head(decoded)


def clean_state_dict(state):
    while state and all(key.startswith("module.") for key in state):
        state = {
            key.removeprefix("module."): value
            for key, value in state.items()
        }
    return state


def checkpoint_state(payload):
    if isinstance(payload, dict):
        for key in ("model_state", "state_dict", "model_state_dict"):
            if key in payload and isinstance(payload[key], dict):
                return clean_state_dict(payload[key])

        if payload and all(
            isinstance(value, torch.Tensor)
            for value in payload.values()
        ):
            return clean_state_dict(payload)

    raise RuntimeError(
        "Không nhận diện được state_dict trong checkpoint."
    )


def load_checkpoint_model(path):
    try:
        payload = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        payload = torch.load(path, map_location="cpu")

    state = checkpoint_state(payload)

    model = UNet34()
    model.load_state_dict(state, strict=True)
    model = model.to(DEVICE)
    model.eval()

    return model


In [ ]:
# -------------------- Discover model checkpoint(s) in Kaggle Input --------------------
selected_candidates = sorted(
    MODEL_SEARCH_ROOT.rglob("unet34_selected_single_best.pt")
)

if selected_candidates:
    # Ưu tiên selected single best được chọn theo internal CV Dice.
    selected_path = selected_candidates[0]

    if len(selected_candidates) > 1:
        print("WARNING: found multiple selected-single checkpoints:")
        for p in selected_candidates:
            print(" -", p)
        print("Using:", selected_path)

    checkpoint_paths = [selected_path]
    EVAL_MODEL_NAME = "Selected single best"
    EVAL_MODE = "single"

else:
    fold_candidates = sorted(
        MODEL_SEARCH_ROOT.rglob("unet34_best.pt")
    )

    if not fold_candidates:
        pt_files = sorted(MODEL_SEARCH_ROOT.rglob("*.pt"))
        raise FileNotFoundError(
            "Không tìm thấy unet34_selected_single_best.pt "
            "hoặc unet34_best.pt trong Kaggle Input.\n"
            + "\n".join(str(p) for p in pt_files[:20])
        )

    # Nếu không có selected checkpoint, ensemble tất cả fold-best tìm được.
    checkpoint_paths = fold_candidates
    EVAL_MODEL_NAME = f"Equal-weight ensemble ({len(checkpoint_paths)} models)"
    EVAL_MODE = "ensemble"

print("Evaluation mode:", EVAL_MODE)
print("Model:", EVAL_MODEL_NAME)
print("Checkpoint(s):")
for p in checkpoint_paths:
    print(" -", p)

runtime_models = [
    load_checkpoint_model(path)
    for path in checkpoint_paths
]

print(
    f"Loaded {len(runtime_models)} model(s) on "
    f"{torch.cuda.get_device_name(0)}"
)


In [ ]:
# -------------------- GPU evaluation: per-image Dice / IoU / Sensitivity --------------------
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    device=DEVICE,
    dtype=torch.float16 if USE_AMP else torch.float32,
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    device=DEVICE,
    dtype=torch.float16 if USE_AMP else torch.float32,
).view(1, 3, 1, 1)


def prepare_inputs(inputs):
    dtype = torch.float16 if USE_AMP else torch.float32
    inputs = inputs.to(
        DEVICE,
        non_blocking=True,
        dtype=dtype,
    )
    inputs = inputs.div(255.0)
    inputs = inputs.sub(IMAGENET_MEAN).div(IMAGENET_STD)
    return inputs.contiguous()


def per_image_metrics(probability, target, threshold=0.5):
    prediction = probability > threshold
    target = target > 0.5

    dims = (1, 2, 3)

    intersection = (prediction & target).sum(dims).float()
    prediction_sum = prediction.sum(dims).float()
    target_sum = target.sum(dims).float()

    union = prediction_sum + target_sum - intersection

    dice = torch.where(
        prediction_sum + target_sum > 0,
        2.0 * intersection
        / (prediction_sum + target_sum).clamp(min=1.0),
        torch.ones_like(intersection),
    )

    jaccard = torch.where(
        union > 0,
        intersection / union.clamp(min=1.0),
        torch.ones_like(intersection),
    )

    sensitivity = torch.where(
        target_sum > 0,
        intersection / target_sum.clamp(min=1.0),
        torch.ones_like(intersection),
    )

    return dice, jaccard, sensitivity


rows = []
prediction_by_id = {}

torch.cuda.synchronize()
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()

with torch.inference_mode():
    for inputs, targets, image_ids in tqdm(
        validation_loader,
        desc="Validation inference",
    ):
        inputs_gpu = prepare_inputs(inputs)
        targets_gpu = targets.to(
            DEVICE,
            non_blocking=True,
        )

        probability_sum = None

        for model in runtime_models:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                probability = torch.sigmoid(
                    model(inputs_gpu).float()
                )

            if probability_sum is None:
                probability_sum = probability
            else:
                probability_sum = probability_sum + probability

        probability = probability_sum / len(runtime_models)

        dice, jaccard, sensitivity = per_image_metrics(
            probability,
            targets_gpu,
            threshold=PRED_THRESHOLD,
        )

        prediction = (
            probability[:, 0] > PRED_THRESHOLD
        ).to(torch.uint8)

        dice_cpu = dice.cpu().numpy()
        jaccard_cpu = jaccard.cpu().numpy()
        sensitivity_cpu = sensitivity.cpu().numpy()
        prediction_cpu = prediction.cpu().numpy()

        for index, image_id in enumerate(image_ids):
            rows.append({
                "image_id": image_id,
                "dice": float(dice_cpu[index]),
                "jaccard": float(jaccard_cpu[index]),
                "sensitivity": float(sensitivity_cpu[index]),
            })
            prediction_by_id[image_id] = prediction_cpu[index]

end.record()
torch.cuda.synchronize()
elapsed_seconds = start.elapsed_time(end) / 1000.0

metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df.sort_values(
    ["dice", "image_id"],
    ascending=[False, True],
).reset_index(drop=True)

csv_path = OUTPUT_DIR / "validation_per_image_metrics.csv"
metrics_df.to_csv(csv_path, index=False)

summary_df = pd.DataFrame([{
    "model": EVAL_MODEL_NAME,
    "n_images": len(metrics_df),
    "mean_dice": metrics_df["dice"].mean(),
    "mean_jaccard": metrics_df["jaccard"].mean(),
    "mean_sensitivity": metrics_df["sensitivity"].mean(),
    "inference_seconds": elapsed_seconds,
    "images_per_second": len(metrics_df) / elapsed_seconds,
}])

summary_path = OUTPUT_DIR / "validation_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df.style.format({
    "mean_dice": "{:.4f}",
    "mean_jaccard": "{:.4f}",
    "mean_sensitivity": "{:.4f}",
    "inference_seconds": "{:.2f}",
    "images_per_second": "{:.1f}",
}))

print("Per-image CSV:", csv_path)
print("Summary CSV  :", summary_path)


In [ ]:
# -------------------- Top 3 best + Top 3 worst by Dice --------------------
top3 = metrics_df.nlargest(3, "dice").copy()
bottom3 = metrics_df.nsmallest(3, "dice").copy()

ranking_df = pd.concat(
    [
        top3.assign(group="Top 3 best"),
        bottom3.assign(group="Top 3 worst"),
    ],
    ignore_index=True,
)

display(
    ranking_df[
        ["group", "image_id", "dice", "jaccard", "sensitivity"]
    ].style.format({
        "dice": "{:.4f}",
        "jaccard": "{:.4f}",
        "sensitivity": "{:.4f}",
    })
)


record_by_id = {
    record["image_id"]: record
    for record in validation_records
}


def read_original_rgb(path):
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def read_resized_gt(path):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(path)
    mask = (mask > 127).astype(np.uint8)
    return cv2.resize(
        mask,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_NEAREST,
    )


visual_rows = [
    ("BEST #1", top3.iloc[0]),
    ("BEST #2", top3.iloc[1]),
    ("BEST #3", top3.iloc[2]),
    ("WORST #1", bottom3.iloc[0]),
    ("WORST #2", bottom3.iloc[1]),
    ("WORST #3", bottom3.iloc[2]),
]

fig, axes = plt.subplots(
    nrows=6,
    ncols=3,
    figsize=(13, 24),
)

for row_index, (rank_label, metric_row) in enumerate(visual_rows):
    image_id = metric_row["image_id"]
    record = record_by_id[image_id]

    original = read_original_rgb(record["image_path"])
    ground_truth = read_resized_gt(record["mask_path"])
    prediction = prediction_by_id[image_id]

    axes[row_index, 0].imshow(original)
    axes[row_index, 0].set_title(
        f"{rank_label} | {image_id}\nDice={metric_row['dice']:.4f}"
    )

    axes[row_index, 1].imshow(
        ground_truth,
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[row_index, 1].set_title("Ground truth")

    axes[row_index, 2].imshow(
        prediction,
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[row_index, 2].set_title("Prediction")

    for column_index in range(3):
        axes[row_index, column_index].axis("off")

fig.suptitle(
    f"{EVAL_MODEL_NAME} — Official Validation\n"
    "Top 3 best and Top 3 worst by per-image Dice",
    fontsize=16,
)
plt.tight_layout(rect=[0, 0, 1, 0.98])

figure_path = OUTPUT_DIR / "top3_best_worst_by_dice.png"
fig.savefig(
    figure_path,
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Visualization:", figure_path)


## Output

Sau khi chạy xong, các file nằm tại:

- `/kaggle/working/validation_eval/validation_summary.csv`
- `/kaggle/working/validation_eval/validation_per_image_metrics.csv`
- `/kaggle/working/validation_eval/top3_best_worst_by_dice.png`

`validation_per_image_metrics.csv` chứa Dice / Jaccard / Sensitivity của **từng ảnh validation**, nên có thể dùng tiếp để phân tích failure cases.